In [1]:
from pathlib import Path
import pygimli as pg  # Stelle sicher, dass das Modul pg importiert ist und verfügbar ist
from pygimli.physics import ert
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime
import os
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.ticker as mticker

from Hilfsfunktionen import T_corr_nach_Inversion
from Hilfsfunktionen import plotting_function
from Hilfsfunktionen import plotting_function_FTL

# Read data
file = "all_timesteps.ohm"
base_dir = Path("filtered_data")
data = []
for unterordner in base_dir.iterdir():
    if unterordner.is_dir():
        datei_pfad = unterordner / "all_timesteps.ohm" 
        if datei_pfad.exists():
            daten_objekt = pg.load(str(datei_pfad))
            data.append([unterordner.name, daten_objekt])
            print(f"Load data: {datei_pfad}")

# Create method manager
manager = []
for ts in data:
    manager.append(ert.ERTManager(ts[1], verbose=True))

[NbConvertApp] Converting notebook Hilfsfunktionen.ipynb to script
[NbConvertApp] Writing 24885 bytes to Hilfsfunktionen.py


Load data: filtered_data\230719\all_timesteps.ohm
Load data: filtered_data\230816\all_timesteps.ohm
Load data: filtered_data\231025\all_timesteps.ohm
Load data: filtered_data\231122\all_timesteps.ohm
Load data: filtered_data\240124\all_timesteps.ohm
Load data: filtered_data\240214\all_timesteps.ohm
Load data: filtered_data\240315\all_timesteps.ohm
Load data: filtered_data\240417\all_timesteps.ohm
Load data: filtered_data\240605\all_timesteps.ohm
Load data: filtered_data\240610\all_timesteps.ohm
Load data: filtered_data\240704\all_timesteps.ohm
Load data: filtered_data\240725\all_timesteps.ohm
Load data: filtered_data\240821\all_timesteps.ohm
Load data: filtered_data\241001\all_timesteps.ohm
Load data: filtered_data\241030\all_timesteps.ohm


In [ ]:
# Ergebnis-Container für alle Indizes
chi2_dict = {}
phi_m_dict = {}
phi_d_dict = {}
rms_dict = {}
rrms_dict = {}

# Für alle Indizes in manager
for idx in range(len(manager)):
    chi2_dict[idx] = []
    phi_m_dict[idx] = []
    phi_d_dict[idx] = []
    rms_dict[idx] = []
    rrms_dict[idx] = []
    
    for lam in lam_list:
        manager[idx].invert(
            quality=34,
            paraMaxCellSize=0.5,
            maxIter=20,
            dPhi=0.1,
            paraDepth=15,
            lam=lam
        )
        chi2_dict[idx].append(manager[idx].inv.chi2())
        phi_d_dict[idx].append(manager[idx].inv.phiData())
        phi_m_dict[idx].append(manager[idx].inv.phiModel())
        rrms_dict[idx].append(round(manager[idx].inv.relrms(), 2))
        rms_dict[idx].append(round(manager[idx].inv.absrms(), 2))


In [ ]:
# alle Kurven plotten
for i in range(len(manager)):
    plt.plot(phi_m_dict[i], phi_d_dict[i], color='grey', marker='o', alpha=0.5)

# Mittelwerte berechnen
phi_m_mean = np.mean([phi_m_dict[i] for i in range(len(manager))], axis=0)
phi_d_mean = np.mean([phi_d_dict[i] for i in range(len(manager))], axis=0)

# Mittelkurve als Linie
plt.plot(phi_m_mean, phi_d_mean, color='red', linewidth=2, label='Mean L-curve')

# Punkte der Mittelkurve einfärben nach lam_list
sc = plt.scatter(phi_m_mean, phi_d_mean, 
                 c=lam_list,                # Werte für die Farbskala
                 cmap='jet',
                 norm=mcolors.LogNorm(vmin=min(lam_list), vmax=max(lam_list)),  # LOG-Skala# Colormap (z.B. plasma, inferno, jet, etc.)
                 s=60,                      # Punktgröße
                 edgecolor='black',         # optional: schwarze Umrandung
                 zorder=3)

# Farbskala hinzufügen
cbar = plt.colorbar(sc)
cbar.set_label("λ (Lambda)")

num_ticks = 6  
tick_values = np.logspace(np.log10(min(lam_list)), np.log10(max(lam_list)), num=num_ticks)
cbar.locator = mticker.FixedLocator(tick_values)
cbar.ax.set_yticklabels([f"{val:.3g}" for val in tick_values])  # formatieren, z.B. 3 sign. Stellen
cbar.ax.minorticks_off()

# Achsenbeschriftung und Titel
plt.xlabel("Model roughness Φₘ")
plt.ylabel("Data misfit Φ_d (χ²)")
plt.title("L-curve")
plt.legend()
plt.show()
